# baseline: color + edge features -> scikit-learn classifier (M3)

Hand-crafted features (HSV histogram, Canny edge density, local binary patterns) feed a
scikit-learn classifier. It predicts the coarse targets from `labels.ipynb`:
**occupancy** (empty/occupied) and **color3** (empty/white/black). It does not tell piece
types apart, so it is compared with the CNN at the occupancy / color level, not on all 13
classes.

Milestone: M3 (baseline vs CNN).

**TODO** (see `docs/M3_TODO.md`)
- [ ] `hsv_histogram`
- [ ] `edge_density` (Canny)
- [ ] `lbp_histogram` (skimage or numpy; add the dependency if skimage)
- [ ] `extract_features`
- [ ] `make_estimator` + `BaselineClassifier` (fit / predict_proba / predict / save / load);
      tune hyperparameters on the val sessions
- [ ] `train_baseline`: fit, val evaluation per target, write `results/<run>/baseline.json`
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np

%run ./labels.ipynb

In [ ]:
TARGETS = {"occupancy": ("y_occ", OCCUPANCY), "color3": ("y_color3", COLOR3)}

## Features
Each feature maps one RGB crop (H, W, 3) uint8 to a 1-D vector. Ideas: empty squares are
flat (few edges, one dominant color); white vs black pieces differ in brightness/hue.

In [ ]:
def hsv_histogram(crop: np.ndarray, bins: tuple[int, int, int] = (8, 4, 4)) -> np.ndarray:
    """Normalized 3D HSV histogram of an RGB crop, flattened (prod(bins),)."""
    # Design:
    # - cv2.cvtColor(..., COLOR_RGB2HSV), cv2.calcHist over the 3 channels (OpenCV hue range
    #   is 0..180), divide by the pixel count so the vector sums to 1.
    # Decision for you: maybe use only the lower part of the crop (the square itself) for
    # occupancy and the upper part (the piece body) for color.
    # TODO: implement
    raise NotImplementedError


def edge_density(crop: np.ndarray, low: int = 50, high: int = 150, grid: int = 2) -> np.ndarray:
    """Fraction of Canny edge pixels, overall and per cell of a grid x grid split
    (1 + grid*grid,). Empty squares have few edges; pieces have many.
    """
    # Design: gray -> cv2.Canny(low, high) -> mean of (edges > 0) on the whole crop and on each cell.
    # TODO: implement
    raise NotImplementedError


def lbp_histogram(crop: np.ndarray, points: int = 8, radius: int = 1) -> np.ndarray:
    """Histogram of uniform local binary patterns on the gray crop (points + 2,), sums to 1."""
    # Design: skimage.feature.local_binary_pattern(method="uniform") (adds a dependency)
    # or a small numpy implementation; uniform LBP has points + 2 distinct codes.
    # Decision for you: pick one.
    # TODO: implement
    raise NotImplementedError


FEATURES = {"hsv_hist": hsv_histogram, "edge_density": edge_density, "lbp": lbp_histogram}


def extract_features(crops, features: list[str]) -> np.ndarray:
    """Stack the named features for each crop -> (N, D) float32."""
    # Design: unknown names -> ValueError listing the valid ones; per crop, concatenate the
    # flattened outputs of FEATURES[name] in the given order.
    # TODO: implement
    raise NotImplementedError

## Classifier
StandardScaler + one sklearn model per target, on shared features. Save/load uses pickle;
because the class is defined in a notebook, `%run ./baseline.ipynb` before loading a
pickle (pickle stores the class as `__main__.BaselineClassifier`).

In [ ]:
def make_estimator(kind: str):
    """StandardScaler + the chosen sklearn classifier ("svm", "random_forest", "logistic"),
    with predict_proba enabled. Unknown kind -> ValueError."""
    # Design:
    # - sklearn.pipeline.make_pipeline(StandardScaler(), model).
    # - class_weight="balanced" everywhere: empty squares dominate.
    # - SVM: SVC(probability=True) is deprecated in recent sklearn; wrap an RBF SVC in
    #   CalibratedClassifierCV instead to get probabilities.
    # Decision for you: hyperparameters (C / gamma, n_estimators, ...) tuned on val sessions.
    # TODO: implement
    raise NotImplementedError


class BaselineClassifier:
    """One sklearn model per target ("occupancy", "color3") on shared features."""

    def __init__(self, features: list[str], classifier: str = "svm",
                 targets: tuple[str, ...] = ("occupancy", "color3")):
        # Design: validate targets against TARGETS (ValueError), store features, classifier,
        # targets and an empty dict models (target -> fitted estimator).
        # TODO: implement
        raise NotImplementedError

    def featurize(self, crops) -> np.ndarray:
        """extract_features with self.features."""
        # TODO: implement
        raise NotImplementedError

    def fit(self, crops, y: dict[str, np.ndarray]) -> "BaselineClassifier":
        """y maps target name -> int labels (e.g. df["y_occ"]). Features are computed once."""
        # Design: featurize once (the expensive part), then one make_estimator(...).fit per target.
        # TODO: implement
        raise NotImplementedError

    def predict_proba(self, crops, target: str = "color3") -> np.ndarray:
        """(N, len(OCCUPANCY)) or (N, len(COLOR3)) probabilities for one target.

        Columns follow the class order in OCCUPANCY / COLOR3. Note: this is not the
        (N, 13) model contract; the baseline cannot see piece types.
        """
        # Design note: if a class is missing from the training data sklearn drops its column;
        # re-insert zeros using self.models[target].classes_ (optional).
        # TODO: implement
        raise NotImplementedError

    def predict(self, crops, target: str = "color3") -> np.ndarray:
        # TODO: implement
        raise NotImplementedError

    def save(self, path: str | Path) -> None:
        """pickle.dump self (create the parent folder)."""
        # TODO: implement
        raise NotImplementedError

    @staticmethod
    def load(path: str | Path) -> "BaselineClassifier":
        """pickle.load; TypeError if the file holds something else."""
        # TODO: implement
        raise NotImplementedError

## Training entry point
Called from `run_train_baseline.ipynb`. Needs `dataset.ipynb` (`%run` it there; it is
not pulled in here to keep this notebook light).

In [ ]:
def train_baseline(cfg, out_path: str | Path = "models/baseline/baseline.pkl") -> BaselineClassifier:
    """Fit the baseline on cfg.data.train_sessions, report val accuracy, save to out_path.

    Requires read_crop / split_by_session from dataset.ipynb in the namespace.
    """
    # Design:
    # - split_by_session(cfg); empty train split -> ValueError.
    # - BaselineClassifier(cfg.baseline.features, cfg.baseline.classifier, cfg.baseline.targets).
    # - Load the crops with read_crop, fit with {target: df[TARGETS[target][0]]}.
    # - Evaluate on splits["val"] per target (accuracy, confusion matrix) and write the numbers
    #   where evaluate.ipynb can pick them up (results/<run_name>/baseline.json); save the pickle.
    # TODO: implement
    raise NotImplementedError

## Self-check
(was tests/test_baseline.py)

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# Plumbing first: register a temporary feature FEATURES["mean_rgb"] = mean RGB / 255 (remove
# it in a finally block) and fake data (dark crops = empty, bright = white piece,
# red = black piece, ~60 small noisy crops):
# - extract_features with an unknown name, BaselineClassifier with target "pieces",
#   make_estimator("xgboost") each raise ValueError.
# - For each kind in svm / random_forest / logistic: fit, predict_proba(color3) has shape
#   (6, 3) with rows summing to 1, occupancy gives (6, 2), training accuracy > 0.9,
#   save + load in a tempdir gives identical predictions.
# Features:
# - hsv_histogram: shape prod(bins), sums to 1; a pure-red crop puts all mass in one hue bin.
# - edge_density: flat crop -> all zeros; checkerboard pattern -> high density; shape 1 + grid*grid.
# - lbp_histogram: shape points + 2, sums to 1; constant image puts all mass in one bin.
# - train_baseline end to end on a tiny fake session with real crop PNGs writes the pickle.
print("baseline: self-check not written yet (see plan above)")